In [24]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [25]:
%config Completer.use_jedi = True

In [8]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100]))

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)


#### Experiment 1: Train Logistic Regression Model

In [10]:
from sklearn.linear_model import LogisticRegression
log_reg = LogisticRegression(C=1, solver='liblinear')
log_reg.fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.96      0.95       270
           1       0.60      0.50      0.55        30

    accuracy                           0.92       300
   macro avg       0.77      0.73      0.75       300
weighted avg       0.91      0.92      0.91       300



#### Experiment 2: Train Random Forect Classifier

In [11]:
from sklearn.ensemble import RandomForestClassifier
rf_clf = RandomForestClassifier(n_estimators=30, max_depth=1)
rf_clf.fit(X_train, y_train)
y_pred = rf_clf.predict(X_test)
print(classification_report(y_test, y_pred))                              
                        

              precision    recall  f1-score   support

           0       0.94      0.99      0.96       270
           1       0.78      0.47      0.58        30

    accuracy                           0.93       300
   macro avg       0.86      0.73      0.77       300
weighted avg       0.93      0.93      0.93       300



In [12]:
#### Experiment 3: Train XGBoost

In [13]:
from xgboost import XGBClassifier
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train, y_train)
y_pred = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred)) 

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.80      0.87        30

    accuracy                           0.98       300
   macro avg       0.97      0.90      0.93       300
weighted avg       0.98      0.98      0.98       300



### Experiment 4: Handle class imbalance using SMOTETomek and then Train XGBoost

In [14]:
from imblearn.combine import SMOTETomek
smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)

np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619]))

In [15]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train_res, y_train_res)
y_pred = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred)) 

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       270
           1       0.81      0.83      0.82        30

    accuracy                           0.96       300
   macro avg       0.89      0.91      0.90       300
weighted avg       0.96      0.96      0.96       300



In [16]:
models = [
    (
        "Logistic Regression",
        {'C':1, 'solver':'liblinear'},
        LogisticRegression(C=1, solver='liblinear'),
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest Classifier",
        {'n_estimators':30, 'max_depth':1},
        RandomForestClassifier(n_estimators=30, max_depth=1),
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGB Classifier",
        {'use_label_encoder':False, 'eval_metric':'logloss'},
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGB Classifier with SMOTE",
        {'use_label_encoder':False, 'eval_metric':'logloss'},
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [17]:
reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]

    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

In [18]:
reports

[{'0': {'precision': 0.9454545454545454,
   'recall': 0.9629629629629629,
   'f1-score': 0.9541284403669725,
   'support': 270.0},
  '1': {'precision': 0.6,
   'recall': 0.5,
   'f1-score': 0.5454545454545454,
   'support': 30.0},
  'accuracy': 0.9166666666666666,
  'macro avg': {'precision': 0.7727272727272727,
   'recall': 0.7314814814814814,
   'f1-score': 0.749791492910759,
   'support': 300.0},
  'weighted avg': {'precision': 0.9109090909090909,
   'recall': 0.9166666666666666,
   'f1-score': 0.91326105087573,
   'support': 300.0}},
 {'0': {'precision': 0.9436619718309859,
   'recall': 0.9925925925925926,
   'f1-score': 0.9675090252707581,
   'support': 270.0},
  '1': {'precision': 0.875,
   'recall': 0.4666666666666667,
   'f1-score': 0.6086956521739131,
   'support': 30.0},
  'accuracy': 0.94,
  'macro avg': {'precision': 0.909330985915493,
   'recall': 0.7296296296296296,
   'f1-score': 0.7881023387223356,
   'support': 300.0},
  'weighted avg': {'precision': 0.9367957746478873

In [19]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

In [26]:
mlflow.set_tracking_uri(uri="http://127.0.0.1:5000/")
mlflow.set_experiment("Anomaly Detection")

for i, elements in enumerate(models):
    model_name = elements[0]
    params = elements[1]
    model = elements[2]
    report = reports[i]

    with mlflow.start_run(run_name=model_name):
        mlflow.log_param('model_name', model_name)
        mlflow.log_param('params', params)
        mlflow.log_metric("accuracy", report['accuracy'])
        mlflow.log_metric('recall_class_0', report['0']['recall'])
        mlflow.log_metric('recall_class_1', report['1']['recall'])
        mlflow.log_metric('f1-score_macro', report['macro avg']['f1-score'])

        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, 'model')
        else:
            mlflow.sklearn.log_model(model, 'model')
                          




2026/03/09 08:34:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/09 08:34:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/09 08:35:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/2/runs/b8709f5976434ba8bb617f4c68a56ee4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/03/09 08:35:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/09 08:35:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest Classifier at: http://127.0.0.1:5000/#/experiments/2/runs/60a64b2c16654455a6e80d3f1934fa1d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/03/09 08:35:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGB Classifier at: http://127.0.0.1:5000/#/experiments/2/runs/c06b653b32304614b369c5f1481349fd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
🏃 View run XGB Classifier with SMOTE at: http://127.0.0.1:5000/#/experiments/2/runs/b1afc73b53d34bae8d68086cab1293a5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


#### Register The Model

In [31]:
model_name = "XGB_Smote"
run_id = input("Enter Run ID:")
model_uri = f"runs:/{run_id}/model"
result = mlflow.register_model(
    model_uri, model_name
)

Enter Run ID: b1afc73b53d34bae8d68086cab1293a5


Successfully registered model 'XGB_Smote'.
2026/03/09 09:22:08 WARNING mlflow.tracking._model_registry.fluent: Run with id b1afc73b53d34bae8d68086cab1293a5 has no artifacts at artifact path 'model', registering model based on models:/m-2d122c83e8914e92913cf3cc0aadacd9 instead
2026/03/09 09:22:08 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB_Smote, version 1
Created version '1' of model 'XGB_Smote'.


#### Load The Model

In [32]:
model_version = 1
model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

In [37]:
dev_model_uri = f"models:/{model_name}@challenger"
prod_model = 'anomaly_detection'

from mlflow.tracking import MlflowClient

client = MlflowClient()
client.copy_model_version(src_model_uri=dev_model_uri, dst_name=prod_model)


Successfully registered model 'anomaly_detection'.
Copied version '1' of model 'XGB_Smote' to version '1' of model 'anomaly_detection'.


<ModelVersion: aliases=[], creation_timestamp=1773029191325, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1773029191325, metrics=None, model_id=None, name='anomaly_detection', params=None, run_id='b1afc73b53d34bae8d68086cab1293a5', run_link='', source='models:/XGB_Smote/1', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [38]:

model_uri = f"models:/{prod_model}@champion"

loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])